In [1]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate scikit-learn

In [2]:
from huggingface_hub import login
from google.colab import userdata
from huggingface_hub import notebook_login

notebook_login()


In [3]:
from datasets import load_dataset

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
print(ds)
print(ds["train"].features)

for i in range(5):
    print(ds["train"][i])

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})
{'text': Value('string'), 'label': Value('int64')}
{'text': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'label': 0}
{'text': '$CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3', 'label': 0}
{'text': '$CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb', 'label': 0}
{'text': '$ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N', 'label': 0}
{'text': '$FNKO - Funko slides after Piper Jaffray PT cut https://t.co/z37IJmCQzB', 'label': 0}


In [4]:
import torch, json, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
feat = ds["train"].features["label"]
if hasattr(feat, "names"):
    id2label = {i: n for i, n in enumerate(feat.names)}
else:
    id2label = {0: "Bearish", 1: "Bullish", 2: "Neutral"}  # verify with your Stage 1 check
label_words = [id2label[i].capitalize() for i in sorted(id2label)]
print(ds, id2label, label_words)

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
model.eval()

label_token_ids = [tok.encode(w, add_special_tokens=False)[0] for w in label_words]
assert len(set(label_token_ids)) == len(label_token_ids), "label first-tokens collide"
print(label_token_ids, [tok.decode([t]) for t in label_token_ids])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
}) {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'} ['Bearish', 'Bullish', 'Neutral']


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[89685, 33, 88007] ['Bear', 'B', 'Neutral']


In [5]:
PROMPT_DATE = "01 Jan 2026"   # fixed forever; the template otherwise inserts today's date

def build_prompt(text):
    msgs = [
        {"role": "system", "content": "You are a financial sentiment classifier."},
        {"role": "user", "content": f"Classify the sentiment of this finance tweet as exactly one of: {', '.join(label_words)}.\n\nTweet: {text}\n\nAnswer with one word."},
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   date_string=PROMPT_DATE)

print(build_prompt("test")[:200])   # must show "Today Date: 01 Jan 2026"
@torch.no_grad()
def predict(texts, bs=16):
    preds = []
    for i in tqdm(range(0, len(texts), bs)):
        batch = [build_prompt(t) for t in texts[i:i+bs]]
        enc = tok(batch, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        logits = model(**enc).logits[:, -1, :]
        preds.extend(logits[:, label_token_ids].argmax(dim=-1).cpu().tolist())
    return preds

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 Jan 2026

You are a financial sentiment classifier.<|eot_id|><|start_header_id|>user<|


In [8]:
val = ds["validation"]
y_true = val["label"]
y_pred = predict(val["text"])
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="macro")
print("accuracy:", acc, "macro-F1:", f1)
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=label_words))

os.makedirs("results", exist_ok=True)
json.dump({"model": MODEL_ID, "setting": "zero-shot, 4-bit nf4",
           "accuracy": acc, "macro_f1": f1, "y_pred": y_pred},
          open("results/baseline_zero_shot.json", "w"))

  0%|          | 0/150 [00:00<?, ?it/s]

accuracy: 0.542713567839196 macro-F1: 0.556775997623085
[[341   3   3]
 [ 61 384  30]
 [735 260 571]]
              precision    recall  f1-score   support

     Bearish       0.30      0.98      0.46       347
     Bullish       0.59      0.81      0.68       475
     Neutral       0.95      0.36      0.53      1566

    accuracy                           0.54      2388
   macro avg       0.61      0.72      0.56      2388
weighted avg       0.78      0.54      0.55      2388



In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
Xtr = vec.fit_transform(ds["train"]["text"])
clf = LogisticRegression(max_iter=1000).fit(Xtr, ds["train"]["label"])
p = clf.predict(vec.transform(val["text"]))
print("tfidf acc:", accuracy_score(y_true, p), "macro-F1:", f1_score(y_true, p, average="macro"))

tfidf acc: 0.8056951423785594 macro-F1: 0.6967714387673319
